# 02-01 nn.Module로 모델 만들기

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. 01-04의 모델을 클래스로 옮기기

In [ ]:
class CircleClassifier(nn.Module):
    def __init__(self, hidden=16):
        super().__init__()  # 꼭 먼저 호출해요
        self.layer1 = nn.Linear(2, hidden)
        self.layer2 = nn.Linear(hidden, 1)

    def forward(self, x):
        # x: (batch, 2)
        h = torch.relu(self.layer1(x))  # (batch, hidden)
        logit = self.layer2(h)  # (batch, 1)
        return logit.squeeze(1)  # (batch,)

model = CircleClassifier()
print(model)

In [ ]:
x = torch.rand(5, 2)  # (5, 2)
print(model(x).shape)  # forward를 직접 부르지 않고 model(x)로 불러요

## 2. 파라미터 살펴보기

In [ ]:
for name, p in model.named_parameters():
    print(f"{name:15s} {tuple(p.shape)}  requires_grad={p.requires_grad}")

total = sum(p.numel() for p in model.parameters())
print("전체 파라미터 수:", total)

## 3. GPU로 옮기고 옵티마이저로 학습하기

In [ ]:
torch.manual_seed(0)
X = torch.rand(400, 2) * 4 - 2  # (400, 2)
Y = ((X ** 2).sum(dim=1) < 1.5).float()  # (400,)
X, Y = X.to(device), Y.to(device)

torch.manual_seed(0)
model = CircleClassifier().to(device)  # 파라미터가 모두 GPU로 옮겨져요
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

losses = []
for step in range(301):
    loss = F.binary_cross_entropy_with_logits(model(X), Y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 100 == 0:
        print(f"step {step:3d}: loss={loss.item():.4f}")

## 4. 학습된 레이어 들여다보기

In [ ]:
g = torch.linspace(-2, 2, 200)
gx, gy = torch.meshgrid(g, g, indexing="xy")
pts = torch.stack([gx.flatten(), gy.flatten()], 1).to(device)  # (40000, 2)
with torch.no_grad():
    prob = torch.sigmoid(model(pts)).reshape(200, 200).cpu()  # (200, 200)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].contourf(gx, gy, prob, levels=20, cmap="coolwarm", alpha=0.6)
axes[0].scatter(X[:, 0].cpu(), X[:, 1].cpu(), c=Y.cpu(), cmap="coolwarm", edgecolor="k", s=8)
axes[0].set_aspect("equal")
axes[0].set_title("decision boundary")
axes[1].plot(losses, color="tab:red")
axes[1].set_title("loss (Adam)")
axes[1].set_xlabel("step")
im = axes[2].imshow(model.layer1.weight.detach().cpu().T, cmap="RdBu_r")
plt.colorbar(im, ax=axes[2])
axes[2].set_title("layer1.weight (transposed)")
axes[2].set_xlabel("hidden unit (16)")
axes[2].set_ylabel("input (2)")
plt.show()

## 5. 저장하고 불러오기

In [ ]:
torch.save(model.state_dict(), "circle.pt")
loaded = CircleClassifier().to(device)
loaded.load_state_dict(torch.load("circle.pt"))
print(torch.allclose(loaded(X), model(X)))